# Сбор, фильтрация и первичная подготовка данных по облигациям

Импорт необходимых библиотек для работы с данными, файловой системой и HTTP-запросами. Создание папок (`history` и `cashflows`) для сохранения выгружаемых данных и настройка сессии `requests` с заголовком `User-Agent` для корректной работы с API Московской биржи.

In [ ]:
import glob
import os
import time
from io import StringIO

import pandas as pd
import requests

os.makedirs("notebooks/rate_sensitivity/bond_shock/history", exist_ok=True)
os.makedirs("notebooks/rate_sensitivity/bond_shock/cashflows", exist_ok=True)

s = requests.Session()
s.headers["User-Agent"] = "Mozilla/5.0"

Чтение исходного файла с биржевыми данными `bonds.csv` с учетом специфики (разделитель: точка с запятой, кодировка `cp1251` для корректной работы с кириллицей и пропуск первых двух строк служебных метаданных). Вывод первых строк таблицы для проверки структуры и корректности парсинга.

In [ ]:
df = pd.read_csv("notebooks/rate_sensitivity/bonds.csv", sep=";", encoding="cp1251", skiprows=2)
df.head()

,SECID,SHORTNAME,ISIN,FACEVALUE,FACEUNIT,ISSUESIZE,MATDATE,YIELDCLOSE,COUPONPERCENT,STARTDATEMOEX,COUPON_BENCHMARK,COUPON_BENCHMARK_SPREAD,OFFERDATE,EMITTER_CREDIT_RATING,LIQUIDITY
0,XS0088543193,RUS-28,XS0088543193,"1000,0",USD,2.499850e+06,2028-06-24,"8,76","12,7500000000",NaN,NaN,NaN,NaN,NaN,2.0
1,XS0114288789,RUS-30,XS0114288789,"0,035",USD,8.823874e+09,2030-03-31,"6,93","7,5000000000",NaN,NaN,NaN,NaN,NaN,2.0
2,XS0223715920,VTB-35,XS0223715920,"1000,0",USD,2.251300e+04,2035-06-30,NaN,"6,2500000000",NaN,NaN,NaN,NaN,NaN,NaN
3,RU000A0JQAM6,ДОМ.РФ15об,RU000A0JQAM6,"1000,0",RUB,7.000000e+06,2028-09-15,NaN,"16,5000000000",NaN,RREFKEYR,"2,5",NaN,AAA,1.0
4,XS0609017917,RZD-31 GBP,XS0609017917,"1000,0",GBP,6.273200e+05,2031-03-25,NaN,"7,4870000000",NaN,NaN,NaN,NaN,NaN,NaN


In [3]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 4030 entries, 0 to 4029
Data columns (total 15 columns):
 #   Column                   Non-Null Count  Dtype  
---  ------                   --------------  -----  
 0   SECID                    4030 non-null   str    
 1   SHORTNAME                4030 non-null   str    
 2   ISIN                     4030 non-null   str    
 3   FACEVALUE                4030 non-null   str    
 4   FACEUNIT                 4030 non-null   str    
 5   ISSUESIZE                3270 non-null   float64
 6   MATDATE                  3968 non-null   str    
 7   YIELDCLOSE               1694 non-null   str    
 8   COUPONPERCENT            3070 non-null   str    
 9   STARTDATEMOEX            3142 non-null   str    
 10  COUPON_BENCHMARK         637 non-null    str    
 11  COUPON_BENCHMARK_SPREAD  636 non-null    str    
 12  OFFERDATE                453 non-null    str    
 13  EMITTER_CREDIT_RATING    3281 non-null   str    
 14  LIQUIDITY                2827 non-n

Очистка и конвертация числовых полей (`FACEVALUE`, `YIELDCLOSE`, `COUPONPERCENT`) с заменой запятых на точки, а также приведение дат к формату `datetime`. Фильтрация датасета по заданным критериям: отбор рублевых инструментов (`FACEUNIT == 'RUB'`), исключение бумаг с плавающим купоном и офертами, проверка наличия дат погашения и положительного купона, а также фильтрация по уровню ликвидности. Формирование списка тикеров для дальнейшей выгрузки котировок.

In [4]:
for c in ["FACEVALUE", "YIELDCLOSE", "COUPONPERCENT"]:
    df[c] = pd.to_numeric(df[c].astype(str).str.replace(",", "."), errors="coerce")
for c in ["MATDATE", "OFFERDATE"]:
    df[c] = pd.to_datetime(df[c], errors="coerce")

sel = df[
    (df["FACEUNIT"] == "RUB")
    & df["COUPON_BENCHMARK"].isna()
    & df["MATDATE"].notna()
    & df["COUPONPERCENT"].notna()
    & (df["COUPONPERCENT"] > 0)
    & df["OFFERDATE"].isna()
    & (df["LIQUIDITY"] == 2)
].copy()

sel = sel[
    ["SECID", "SHORTNAME", "MATDATE", "COUPONPERCENT", "EMITTER_CREDIT_RATING"]
].reset_index(drop=True)
print(len(sel))

secids = sel["SECID"].sample(min(150, len(sel)), random_state=42).tolist()

761


Формирование для запроса данных по ключевой ставке ЦБ РФ с 2013 года по текущую дату. Парсинг табличных данных из HTML-ответа с помощью `pandas`, очистка, переименование колонок, приведение даты к формату `datetime`, а ставки - к числовому типу. Расчет значения ставки в базисных пунктах (`key_rate_bp`), сохранение готовой истории в файл `bond_shock/key_rate.csv`.

In [6]:
d_to = pd.Timestamp.today().strftime("%d.%m.%Y")
url = f"https://www.cbr.ru/hd_base/KeyRate/?UniDbQuery.Posted=True&UniDbQuery.From=01.01.2013&UniDbQuery.To={d_to}"

r = s.get(url, timeout=30)
r.raise_for_status()

key_rate = pd.read_html(StringIO(r.text), decimal=",", thousands=None)[0].iloc[:, :2]
key_rate.columns = ["date", "key_rate_pct"]
key_rate["date"] = pd.to_datetime(key_rate["date"], format="%d.%m.%Y")
key_rate["key_rate_pct"] = key_rate["key_rate_pct"].astype(float)
key_rate["key_rate_bp"] = key_rate["key_rate_pct"] * 100
key_rate = key_rate.sort_values("date").reset_index(drop=True)

print(key_rate.describe())

key_rate.to_csv("bond_shock/key_rate.csv", index=False)
key_rate.head()

                             date  key_rate_pct  key_rate_bp
count                        3286   3286.000000  3286.000000
mean   2020-04-22 09:01:12.306756     10.433658  1043.365794
min           2013-09-17 00:00:00      4.250000   425.000000
25%           2017-01-12 06:00:00      7.250000   725.000000
50%           2020-05-05 12:00:00      8.500000   850.000000
75%           2023-08-03 18:00:00     14.000000  1400.000000
max           2026-10-23 00:00:00     21.000000  2100.000000
std                           NaN      4.625681   462.568149


,date,key_rate_pct,key_rate_bp
0,2013-09-17,5.5,550.0
1,2013-09-18,5.5,550.0
2,2013-09-19,5.5,550.0
3,2013-09-20,5.5,550.0
4,2013-09-23,5.5,550.0


Определение вспомогательных функций для работы с API Московской биржи. Функция `iss_table` реализует обход ограничения на количество строк с помощью параметра `start`, обработку JSON-ответов и защиту от бесконечного цикла. Функции `load_history` и `load_cashflows` используют её для скачивания исторических котировок, купонных выплат и графиков амортизации по выбранным облигациям.

In [7]:
def iss_table(path, table, params=None):
    params = dict(params or {})
    params["iss.meta"] = "off"
    params["iss.only"] = table
    pages = []
    start = 0
    prev = None
    while True:
        params["start"] = start
        r = s.get(f"https://iss.moex.com/iss/{path}.json", params=params, timeout=30)
        r.raise_for_status()
        block = r.json().get(table, {"data": [], "columns": []})
        if not block["data"]:
            break
        page = pd.DataFrame(block["data"], columns=block["columns"])
        if prev is not None and page.equals(prev):
            break
        pages.append(page)
        prev = page
        start += len(page)
        time.sleep(0.15)
    if pages:
        return pd.concat(pages, ignore_index=True)
    return pd.DataFrame()


def load_history(secid):
    return iss_table(
        f"history/engines/stock/markets/bonds/securities/{secid}",
        "history",
        {"from": "2014-01-01"},
    )


def load_cashflows(secid):
    cpn = iss_table(f"securities/{secid}/bondization", "coupons")
    amo = iss_table(f"securities/{secid}/bondization", "amortizations")
    return cpn, amo

Тестирование написанных функций на примере первой облигации из сформированного списка. Загрузка истории торгов, купонных выплат и графиков амортизации, а также вывод информации о размерах таблиц, списках колонок и последних строках для каждого датасета с целью контроля качества данных.

In [8]:
s0 = secids[0]
h = load_history(s0)
cpn, amo = load_cashflows(s0)

print(s0, h.shape)
print(h.columns.tolist())
print(h.tail(3).T)
print(cpn.columns.tolist())
print(cpn.tail(3).T)
print(amo.columns.tolist())
print(amo.tail(3).T)

RU000A10BB75 (392, 50)
['BOARDID', 'TRADEDATE', 'SHORTNAME', 'SECID', 'NUMTRADES', 'VALUE', 'LOW', 'HIGH', 'CLOSE', 'LEGALCLOSEPRICE', 'ACCINT', 'WAPRICE', 'YIELDCLOSE', 'OPEN', 'VOLUME', 'MARKETPRICE2', 'MARKETPRICE3', 'ADMITTEDQUOTE', 'MP2VALTRD', 'MARKETPRICE3TRADESVALUE', 'ADMITTEDVALUE', 'MATDATE', 'DURATION', 'YIELDATWAP', 'IRICPICLOSE', 'BEICLOSE', 'COUPONPERCENT', 'COUPONVALUE', 'BUYBACKDATE', 'LASTTRADEDATE', 'FACEVALUE', 'CURRENCYID', 'CBRCLOSE', 'YIELDTOOFFER', 'YIELDLASTCOUPON', 'OFFERDATE', 'FACEUNIT', 'TRADINGSESSION', 'CALLOPTIONDATE', 'CALLOPTIONYIELD', 'CALLOPTIONDURATION', 'PUTOPTIONDATE', 'DATEYIELDFROMISSUER', 'TRADE_SESSION_DATE', 'ZSPREAD', 'ZSPREADATWAPRICE', 'BONDTYPE', 'BONDSUBTYPE', 'COUPON_DETAILS', 'FACEVALUE_TYPE']
                                                                   389  \
BOARDID                                                           TQRD   
TRADEDATE                                                   2026-10-21   
SHORTNAME               

Итеративный обход всего списка отобранных облигаций с проверкой локального кэша, если файлы истории, купонов и амортизации уже существуют, запрос пропускается. Загрузка данных через MOEX ISS API с сохранением в CSV-файлы в директории `bond_shock/`. Реализована обработка исключений с записью неудачных попыток в список `failed` и вывод прогресса каждые 10 итераций.

In [ ]:
failed = []

for i, secid in enumerate(secids, 1):
    hp = f"notebooks/rate_sensitivity/bond_shock/history/{secid}.csv"
    cp = f"notebooks/rate_sensitivity/bond_shock/cashflows/{secid}_cpn.csv"
    ap = f"notebooks/rate_sensitivity/bond_shock/cashflows/{secid}_amo.csv"

    if os.path.exists(hp) and os.path.exists(cp) and os.path.exists(ap):
        continue

    try:
        load_history(secid).to_csv(hp, index=False)
        c, a = load_cashflows(secid)
        c.to_csv(cp, index=False)
        a.to_csv(ap, index=False)
    except Exception as e:  # noqa: BLE001
        failed.append((secid, repr(e)))

    if i % 10 == 0:
        print(i, "/", len(secids))

print("не скачались:", failed)

10 / 150
20 / 150
30 / 150
40 / 150
50 / 150
60 / 150
70 / 150
80 / 150
90 / 150
100 / 150
110 / 150
120 / 150
130 / 150
140 / 150
150 / 150
не скачались: []


Поиск всех скачанных CSV-файлов с историей котировок в директории `bond_shock/history/`, фильтрация пустых файлов и объединение их в единый DataFrame. Сохранение объединенного датасета в файл `full_bonds_dataset.csv` и вывод итогового размера полученной таблицы (количество строк и колонок).

In [ ]:
history_files = glob.glob("notebooks/rate_sensitivity/bond_shock/history/*.csv")
df_list = [pd.read_csv(f) for f in history_files if os.path.getsize(f) > 0]
full_df = pd.concat(df_list, ignore_index=True)
output_path = "notebooks/rate_sensitivity/bond_shock/full_bonds_dataset.csv"
full_df.to_csv(output_path, index=False)
print(f"Строк: {len(full_df)}, колонок: {len(full_df.columns)}")

Строк: 90287, колонок: 50


In [11]:
full_df.info()

<class 'pandas.DataFrame'>
RangeIndex: 90287 entries, 0 to 90286
Data columns (total 50 columns):
 #   Column                   Non-Null Count  Dtype  
---  ------                   --------------  -----  
 0   BOARDID                  90287 non-null  str    
 1   TRADEDATE                90287 non-null  str    
 2   SHORTNAME                90287 non-null  str    
 3   SECID                    90287 non-null  str    
 4   NUMTRADES                90287 non-null  int64  
 5   VALUE                    90287 non-null  float64
 6   LOW                      82590 non-null  float64
 7   HIGH                     82590 non-null  float64
 8   CLOSE                    82590 non-null  float64
 9   LEGALCLOSEPRICE          89547 non-null  float64
 10  ACCINT                   86465 non-null  float64
 11  WAPRICE                  82726 non-null  float64
 12  YIELDCLOSE               83670 non-null  float64
 13  OPEN                     82590 non-null  float64
 14  VOLUME                   90287 no

Приведение колонок с датами к формату `datetime` и обогащение истории торгов данными о ключевой ставке Банка России путем слияния по дате сделки. Добавление информации о кредитном рейтинге эмитента из таблицы метаданных и вывод первых строк итогового объединенного датасета.

In [12]:
full_df["TRADEDATE"] = pd.to_datetime(full_df["TRADEDATE"])
key_rate["date"] = pd.to_datetime(key_rate["date"])

full_df = pd.merge(
    full_df,
    key_rate[["date", "key_rate_pct", "key_rate_bp"]],
    left_on="TRADEDATE",
    right_on="date",
    how="left",
)
full_df.drop(columns=["date"], inplace=True, errors="ignore")

bonds_meta = sel[["SECID", "EMITTER_CREDIT_RATING"]].copy()
full_df = pd.merge(full_df, bonds_meta, on="SECID", how="left")

full_df.head()

C:\Users\dkita\AppData\Local\Temp\ipykernel_9516\3155265315.py:11: Pandas4Warning: The inplace keyword in DataFrame.drop is deprecated and will be removed in a future version (PDEP-8).
See https://pandas.pydata.org/docs/dev/whatsnew/v3.1.0.html#deprecation-inplace for more details.
  full_df.drop(columns=["date"], inplace=True, errors="ignore")


,BOARDID,TRADEDATE,SHORTNAME,SECID,NUMTRADES,VALUE,LOW,HIGH,CLOSE,LEGALCLOSEPRICE,...,TRADE_SESSION_DATE,ZSPREAD,ZSPREADATWAPRICE,BONDTYPE,BONDSUBTYPE,COUPON_DETAILS,FACEVALUE_TYPE,key_rate_pct,key_rate_bp,EMITTER_CREDIT_RATING
0,EQNO,2014-01-06,РЖД-28 обл,RU000A0JTU85,0,0.0,NaN,NaN,NaN,102.39,...,NaN,NaN,NaN,NaN,NaN,Фиксированный с известными купонами,Фиксированный,NaN,NaN,AAA
1,EQNO,2014-01-08,РЖД-28 обл,RU000A0JTU85,0,0.0,NaN,NaN,NaN,102.39,...,NaN,NaN,NaN,NaN,NaN,Фиксированный с известными купонами,Фиксированный,NaN,NaN,AAA
2,EQNO,2014-01-09,РЖД-28 обл,RU000A0JTU85,1,15360000.0,102.4,102.4,102.4,102.40,...,NaN,NaN,NaN,NaN,NaN,Фиксированный с известными купонами,Фиксированный,5.5,550.0,AAA
3,EQNO,2014-01-10,РЖД-28 обл,RU000A0JTU85,0,0.0,NaN,NaN,NaN,102.40,...,NaN,NaN,NaN,NaN,NaN,Фиксированный с известными купонами,Фиксированный,5.5,550.0,AAA
4,EQNO,2014-01-13,РЖД-28 обл,RU000A0JTU85,0,0.0,NaN,NaN,NaN,102.40,...,NaN,NaN,NaN,NaN,NaN,Фиксированный с известными купонами,Фиксированный,5.5,550.0,AAA


In [13]:
full_df.info()

<class 'pandas.DataFrame'>
RangeIndex: 90287 entries, 0 to 90286
Data columns (total 53 columns):
 #   Column                   Non-Null Count  Dtype         
---  ------                   --------------  -----         
 0   BOARDID                  90287 non-null  str           
 1   TRADEDATE                90287 non-null  datetime64[us]
 2   SHORTNAME                90287 non-null  str           
 3   SECID                    90287 non-null  str           
 4   NUMTRADES                90287 non-null  int64         
 5   VALUE                    90287 non-null  float64       
 6   LOW                      82590 non-null  float64       
 7   HIGH                     82590 non-null  float64       
 8   CLOSE                    82590 non-null  float64       
 9   LEGALCLOSEPRICE          89547 non-null  float64       
 10  ACCINT                   86465 non-null  float64       
 11  WAPRICE                  82726 non-null  float64       
 12  YIELDCLOSE               83670 non-null  fl

Формирование списка `cols_to_drop` для удаления служебных признаков, дублирующихся после слияния колонок, технических биржевых метрик, данных по опционам и текстовых идентификаторов, не участвующих в моделях. Удаление этих столбцов из датафрейма с проверкой их наличия, перезапись итогового чистого файла `bond_shock/full_bonds_dataset.csv` и вывод списка оставшихся полезных колонок.

In [ ]:
cols_to_drop = [
    # Технические цены, объемы и биржевые метрики
    "LEGALCLOSEPRICE",
    "MARKETPRICE2",
    "MARKETPRICE3",
    "ADMITTEDQUOTE",
    "ADMITTEDVALUE",
    "MP2VALTRD",
    "MARKETPRICE3TRADESVALUE",
    "LOW",
    "HIGH",
    "OPEN",
    "NUMTRADES",
    "VOLUME",
    "VALUE",
    "BOARDID",
    "LASTTRADEDATE",
    # Опционы, оферты и коллбэки
    "BUYBACKDATE",
    "OFFERDATE",
    "CALLOPTIONDATE",
    "CALLOPTIONYIELD",
    "CALLOPTIONDURATION",
    "PUTOPTIONDATE",
    "YIELDTOOFFER",
    "YIELDLASTCOUPON",
    # Служебные индикаторы
    "ZSPREAD",
    "ZSPREADATWAPRICE",
    "IRICPICLOSE",
    "BEICLOSE",
    "CBRCLOSE",
    "DATEYIELDFROMISSUER",
    "TRADE_SESSION_DATE",
    "TRADINGSESSION",
    # Типы и текстовые классификаторы
    "BONDTYPE",
    "BONDSUBTYPE",
    "COUPON_DETAILS",
    "FACEVALUE_TYPE",
    "CURRENCYID",
    "FACEUNIT",
    "key_rate_pct_x",
    "key_rate_bp_x",
    "EMITTER_CREDIT_RATING_x",
    "key_rate_pct_y",
    "key_rate_bp_y",
    "EMITTER_CREDIT_RATING_y",
]

full_df.drop(
    columns=[c for c in cols_to_drop if c in full_df.columns],
    inplace=True,
    errors="ignore",
)
full_df.to_csv(
    "notebooks/rate_sensitivity/bond_shock/full_bonds_dataset.csv", index=False
)

print("Итоговое количество колонок:", len(full_df.columns))
print("Оставшиеся колонки:")
print(full_df.columns.tolist())

C:\Users\dkita\AppData\Local\Temp\ipykernel_9516\2713228383.py:51: Pandas4Warning: The inplace keyword in DataFrame.drop is deprecated and will be removed in a future version (PDEP-8).
See https://pandas.pydata.org/docs/dev/whatsnew/v3.1.0.html#deprecation-inplace for more details.
  full_df.drop(


Итоговое количество колонок: 16
Оставшиеся колонки:
['TRADEDATE', 'SHORTNAME', 'SECID', 'CLOSE', 'ACCINT', 'WAPRICE', 'YIELDCLOSE', 'MATDATE', 'DURATION', 'YIELDATWAP', 'COUPONPERCENT', 'COUPONVALUE', 'FACEVALUE', 'key_rate_pct', 'key_rate_bp', 'EMITTER_CREDIT_RATING']


In [15]:
full_df.info()

<class 'pandas.DataFrame'>
RangeIndex: 90287 entries, 0 to 90286
Data columns (total 16 columns):
 #   Column                 Non-Null Count  Dtype         
---  ------                 --------------  -----         
 0   TRADEDATE              90287 non-null  datetime64[us]
 1   SHORTNAME              90287 non-null  str           
 2   SECID                  90287 non-null  str           
 3   CLOSE                  82590 non-null  float64       
 4   ACCINT                 86465 non-null  float64       
 5   WAPRICE                82726 non-null  float64       
 6   YIELDCLOSE             83670 non-null  float64       
 7   MATDATE                90287 non-null  str           
 8   DURATION               83294 non-null  float64       
 9   YIELDATWAP             82466 non-null  float64       
 10  COUPONPERCENT          85800 non-null  float64       
 11  COUPONVALUE            89281 non-null  float64       
 12  FACEVALUE              90287 non-null  float64       
 13  key_rate_pct

Заполнение пропусков в кредитных рейтингах эмитентов статусом Not Rated и удаление строк с пропущенными значениями в критически важных для анализа колонках. Перезапись очищенного датасета в файл `full_bonds_dataset.csv`, а также вывод итогового количества строк и проверки оставшихся пропусков.

In [ ]:
full_df["EMITTER_CREDIT_RATING"] = full_df["EMITTER_CREDIT_RATING"].fillna("NR")

critical_cols = ["TRADEDATE", "SECID", "CLOSE", "FACEVALUE", "MATDATE", "key_rate_pct"]
full_df.dropna(subset=critical_cols, inplace=True)

full_path = "notebooks/rate_sensitivity/bond_shock/full_bonds_dataset.csv"
full_df.to_csv(full_path, index=False)

print(f"Строк после очистки: {len(full_df)}")
print("Пропуски по колонкам:\n", full_df.isnull().sum())

C:\Users\dkita\AppData\Local\Temp\ipykernel_9516\1978177061.py:4: Pandas4Warning: The inplace keyword in DataFrame.dropna is deprecated and will be removed in a future version (PDEP-8).
See https://pandas.pydata.org/docs/dev/whatsnew/v3.1.0.html#deprecation-inplace for more details.
  full_df.dropna(subset=critical_cols, inplace=True)


Строк после очистки: 82567
Пропуски по колонкам:
 TRADEDATE                   0
SHORTNAME                   0
SECID                       0
CLOSE                       0
ACCINT                   2699
WAPRICE                    89
YIELDCLOSE               3107
MATDATE                     0
DURATION                 3534
YIELDATWAP                349
COUPONPERCENT            2092
COUPONVALUE               505
FACEVALUE                   0
key_rate_pct                0
key_rate_bp                 0
EMITTER_CREDIT_RATING       0
dtype: int64


## Итоги этапа: сборка и анализ датасета

На выходе первого пайплайна мы получили качественный, очищенный датасет `full_bonds_dataset.csv`, содержащий **82 567 строк** исторических биржевых данных.
* **Критические поля** (`TRADEDATE`, `SECID`, `CLOSE`, `FACEVALUE`, `MATDATE`, `key_rate_pct`) полностью очищены и имеют 0 пропусков.
* **Вторичные признаки** (`DURATION`, `YIELDCLOSE`, `ACCINT`, `WAPRICE` и др.) содержат единичные пропуски. Это нормальная ситуация для данных Московской биржи, возникающая в дни без активных торгов, при пересчете купонных периодов или из-за особенностей расчета биржевых индикаторов. Удалять эти строки не требуется, так как алгоритмы градиентного бустинга (CatBoost), которые мы будем использовать на следующем этапе, эффективно обрабатывают пропуски.